# 2.2 — DPI-Flow

**Папка 2, подноутбук 2.** Grid search по гиперпараметрам DPI-Flow с богатой историей по
всем метрикам и **выбором метрики отбора** → сохранение лучших в
`models/dpi_flow/hyperparams.json` → финальное обучение чтением JSON с отслеживанием метрик.

## Окружение и данные

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset, train_model
from liquefaction_ai.training import grid_search, write_hyperparams, read_hyperparams, save_trained_model
from liquefaction_ai.evaluation import METRICS, english_metric_table, metrics_catalog, subsample_split
from liquefaction_ai.viz import grid_search_dashboard, training_dashboard, lines

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
static_dim = benchmark["train"]["static"].shape[1]
prefix_dim = benchmark["train"]["prefix_summary"].shape[1]
seq_dim = benchmark["train"]["seq_in"].shape[-1]

# Grid search выполняется на компактной подвыборке (для ранжирования гиперпараметров).
gs_train = subsample_split(benchmark["train"], 2000, config.seed)
gs_val = subsample_split(benchmark["val"], 600, config.seed + 1)


def show_grid_dashboard(res, grid, score, metric_keys, fig_id):
    """Построить дашборд grid search: по Y — метрики, по X — текст конфигурации."""
    info = METRICS[score]
    labels = {k: f"{METRICS[k].name} ({METRICS[k].units})" for k in metric_keys}
    fmts = {k: METRICS[k].fmt for k in metric_keys}
    return grid_search_dashboard(res, metric_keys, list(grid.keys()), score,
                                 metric_labels=labels, metric_fmts=fmts,
                                 lower_is_better=info.lower_is_better, target=info.target,
                                 save=SAVE_FIGS, fig_id=fig_id)

print("device:", device, "| dims static/prefix/seq:", static_dim, prefix_dim, seq_dim)
from liquefaction_ai.models import DPIFlow
from liquefaction_ai.evaluation import collect_outputs

device: cpu | dims static/prefix/seq: 34 6 5


## Каталог метрик

Все метрики качества определены с подробными описаниями в `liquefaction_ai.evaluation.metrics`
(`METRICS`) и импортируются в ноутбук. **Метрику отбора лучших гиперпараметров можно выбрать**
через переменную `SELECTION_METRIC` ниже.

In [2]:
display(metrics_catalog())

,Metric,Name,Units,Direction,Description
0,val_loss,Validation loss,–,lower is better,Mean validation value of the model's training ...
1,Traj_RMSE,Trajectory RMSE,–,lower is better,Root-mean-square error of the predicted pore-p...
2,Traj_MAE,Trajectory MAE,–,lower is better,Mean absolute error of the predicted PPR(N) tr...
3,Traj_MSE,Trajectory MSE,–,lower is better,Mean squared error of the predicted PPR(N) tra...
4,N_liq_MAE,MAE of N_liq,cycles,lower is better,Mean absolute error of the predicted number of...
5,AUROC,AUROC,–,higher is better,Area under the ROC curve for liquefaction-risk...
6,AUPRC,AUPRC,–,higher is better,Area under the precision–recall curve; classif...
7,Brier,Brier score,–,lower is better,Mean squared error of the predicted liquefacti...
8,ECE,Expected calibration error,–,lower is better,Average absolute gap between predicted confide...
9,Coverage_90,90% interval coverage,–,target ≈ 0.9,Empirical fraction of true PPR values that fal...


## Шаг 1. Grid search и сохранение гиперпараметров

In [3]:
# >>> Метрика, по которой grid search выбирает лучшие гиперпараметры <<<
SELECTION_METRIC = "Traj_RMSE"   # например: "Traj_RMSE", "Brier", "AUROC", "N_liq_MAE", "val_loss"
DASHBOARD_METRICS = ["Traj_RMSE", "AUROC", "Brier", "ECE", "N_liq_MAE", "Coverage_90"]

fixed = dict(static_dim=static_dim, prefix_dim=prefix_dim, seq_len=config.seq_len,
             prefix_len=config.prefix_len, max_cycle_reference=config.max_cycle_reference,
             theta_dim=31, probabilistic=True, use_analytical_layer=True)
grid = {"hidden_dim": [128, 160], "calibration_steps": [1, 2]}
res, best = grid_search(lambda p: DPIFlow(**fixed, **p), grid, gs_train, gs_val,
                        config, device, search_epochs=2, score_metric=SELECTION_METRIC)
print("Selection metric:", SELECTION_METRIC, "| best:", best)
display(english_metric_table(res).round(4))
write_hyperparams(MODELS_DIR, "dpi_flow", {"model_type": "DPIFlow", "display_name": "DPI-Flow",
                  "model_kwargs": {**fixed, **best},
                  "search": {"grid": grid, "score_metric": SELECTION_METRIC, "best": best}})
show_grid_dashboard(res, grid, SELECTION_METRIC, DASHBOARD_METRICS, "2_2_grid_search").show()

Selection metric: Traj_RMSE | best: {'hidden_dim': 128, 'calibration_steps': 2}


,hidden_dim,calibration_steps,val_loss,MAE N_liq (cycles),RMSE N_liq (cycles),log-MAE N_liq,log-RMSE N_liq,AUROC,AUPRC,Brier,...,Interval width@80%,Coverage@90%,Interval width@90%,Coverage@95%,Interval width@95%,Calibration error,Trajectory NLL,Trajectory CRPS,CRR-curve RMSE,Produces CRR
0,128,2,-0.7912,1943.6707,3347.7295,2.7628,3.0448,0.7701,0.8588,0.1927,...,0.4004,0.8387,0.5139,0.9107,0.6123,0.0471,-0.1928,0.1029,0.2029,1.0
1,160,1,-0.6421,2166.7881,3587.4050,2.1327,2.5008,0.3406,0.5179,0.3345,...,0.4128,0.8755,0.5298,0.9021,0.6313,0.0257,-0.1661,0.1010,0.1984,1.0
2,128,1,-0.5530,2340.4241,3734.8813,2.2241,2.6207,0.4453,0.5470,0.3422,...,0.4112,0.8158,0.5277,0.8657,0.6288,0.0710,-0.1137,0.1105,0.2110,1.0
3,160,2,-0.4293,2355.3237,3750.2021,2.2875,2.6815,0.0872,0.4165,0.4016,...,0.4004,0.8485,0.5139,0.8776,0.6123,0.0471,-0.0540,0.1067,0.2012,1.0


[save_figure] PNG для '2_2_grid_search' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Шаг 2. Финальное обучение по сохранённым гиперпараметрам

In [4]:
hp = read_hyperparams(MODELS_DIR, "dpi_flow")
model = DPIFlow(**hp["model_kwargs"]).to(device)
model, history = train_model(model, benchmark["train"], benchmark["val"], epochs=config.physics_epochs,
                             model_name="DPI-Flow", config=config, device=device, track_metrics=True, scheduler="cosine")
save_trained_model(model, MODELS_DIR, "dpi_flow", {**hp, "epochs": config.physics_epochs,
                   "learning_rate": config.learning_rate, "weight_decay": config.weight_decay,
                   "batch_size": config.batch_size, "seed": config.seed}, history)
print("saved:", MODELS_DIR / "dpi_flow")

[DPI-Flow] эпоха 01 | обучение=3.7991 | валидация=0.1425 | val_AUROC=0.351 | val_RMSE=0.2522


[DPI-Flow] эпоха 02 | обучение=-0.1413 | валидация=-0.6355 | val_AUROC=0.308 | val_RMSE=0.1961


[DPI-Flow] эпоха 03 | обучение=-0.6810 | валидация=-0.8715 | val_AUROC=0.477 | val_RMSE=0.1856


[DPI-Flow] эпоха 04 | обучение=-0.9313 | валидация=-1.1020 | val_AUROC=0.640 | val_RMSE=0.1649


[DPI-Flow] эпоха 05 | обучение=-1.1171 | валидация=-1.2230 | val_AUROC=0.800 | val_RMSE=0.1533


[DPI-Flow] эпоха 06 | обучение=-1.2024 | валидация=-1.2930 | val_AUROC=0.846 | val_RMSE=0.1475


saved: /sessions/determined-cool-fermat/mnt/liquefaction-ai/models/dpi_flow


## Кривые обучения с метриками

In [5]:
training_dashboard(history, title="Training dynamics — DPI-Flow", model_color="#0b6efd",
                   save=SAVE_FIGS, fig_id="2_2_training_dashboard").show()

[save_figure] PNG для '2_2_training_dashboard' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Предпросмотр предсказаний

In [6]:
test = benchmark["test"]
outputs = collect_outputs(model, test, config, device)
cycles = test["cycles"].cpu().numpy()
r_true = (test["r_true"] if "r_true" in test else test["r_obs"]).cpu().numpy()  # реальные данные: измеренная кривая
pred = outputs["traj_mean"]
tm = test["meta"].reset_index(drop=True)
pick = tm.sort_values("PPR_max_true", ascending=False).head(2).index.tolist() + tm.sort_values("PPR_max_true").head(2).index.tolist()
series = []
for k, idx in enumerate(pick):
    col = ["#0b6efd", "#198754", "#fd7e14", "#6610f2"][k]
    series.append({"x": cycles[idx], "y": r_true[idx], "name": f"measured #{idx}", "color": col})
    series.append({"x": cycles[idx], "y": pred[idx], "name": f"DPI-Flow #{idx}", "color": col, "dash": "dash"})
lines(series, title="DPI-Flow: true vs predicted PPR trajectories", xlabel="Number of cycles, N",
      ylabel="Pore-pressure ratio, PPR (–)", logx=True, hline=1.0,
      save=SAVE_FIGS, fig_id="2_2_prediction_preview").show()

[save_figure] PNG для '2_2_prediction_preview' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

DPI-Flow подобрана grid search (выбор метрики), обучена и сохранена. Дальше — **2.3 EVT-NeuralSSM**.